# Task 4.3: effect of PARRM on ridge refit

Because unsupervised PARRM artifact removal fails to restore the accuracy of an OFF-trained ridge decoder (Task 3), this notebook tests whether PARRM improves decoding accuracy when refitting on $k$ labeled ON trials. Each session supplies a 20-trial calibration pool and a fixed 30-trial query (Section 4.1), so we refit separate ridge decoders on raw and PARRM-cleaned features with the same draws and the same penalty, which isolates the effect of artifact removal.

## PARRM

During Butters' reaching task, motor cortex local field potentials (0.1–300 Hz) recorded by a 64-electrode array are dominated by a periodic stimulation artifact from a partially demodulated 3.5 kHz carrier modulated at 3 Hz or 20 Hz.

PARRM removes this artifact without labels by phase-averaging across cycles to subtract a waveform template, which neither restores OFF-trained decoders (Task 3.2) nor improves decoders refitted on k labeled ON trials.

Refitting with PARRM fails to help because within-session artifact drift leaves large residuals (up to z ≈ 5300), template subtraction removes genuine phase-locked neural and movement signals, and ridge regression already down-weights artifact-dominated channels on its own.

## Setup

In [ ]:
import os
import sys

HERE = os.path.abspath(os.getcwd())
TASK4_DIR = HERE if os.path.basename(HERE) == 'task4' else os.path.join(HERE, 'task4')
TASK3_DIR = os.path.join(os.path.dirname(TASK4_DIR), 'task3')
for path in (TASK4_DIR, TASK3_DIR):
    if path not in sys.path:
        sys.path.insert(0, path)

import utils
import task3_common  # ridge solver, penalty grid, clipping and PARRM map shared with task 3

print('task 4 helpers:', utils.TASK4_DIR)
print('task 3 helpers:', task3_common.TASK_DIR)
print('draws file    :', utils.DRAWS_PATH)


task 4 helpers: /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4
task 3 helpers: /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task3
draws file    : /Volumes/Kevin's SSD/Classes/CS182/EECS182-Project/task4/results/draws.json


### Reference ridge and kNN decoders

4.3 fits its refits with the task 3 solver (`task3_common.ridge_path`), so these numbers and
task 3's come from the same code. The two classes below are the project's standalone
reference implementations of ridge and kNN: this notebook does not call them, but
`task4/utils.py` re-exports them and the figure scripts use them
(`scripts/make_diagnostics.py`, `scripts/make_parameter_sweeps.py`). The grid constants stay
with them for those sweeps; the refit below uses `task3_common.LAMBDAS`.


In [ ]:
# MODELS: shared with task4/utils.py (the compat shim executes this cell)
import numpy as np
from sklearn.neighbors import KNeighborsRegressor

RIDGE_PENALTIES = (100, 1_000, 10_000, 100_000, 1_000_000, 10_000_000)
KNN_NEIGHBOR_GRID = (5, 10, 25, 50, 100, 200)
CALIBRATION_TUNE_TRIALS = 10  # calibration trials used to tune KNN


class RidgeRegressor:
    """Closed-form ridge on bin rows.

    bin_features (n_bins, n_features), bin_velocity (n_bins, 2).
    """

    def __init__(self, alpha):
        self.alpha = alpha

    def fit(self, bin_features, bin_velocity):
        bin_features = np.asarray(bin_features, np.float64)
        bin_velocity = np.asarray(bin_velocity, np.float64)
        feature_mean = bin_features.mean(axis=0)
        velocity_mean = bin_velocity.mean(axis=0)
        features_centred = bin_features - feature_mean
        velocity_centred = bin_velocity - velocity_mean
        # (X^T X + alpha I) W = X^T Y
        normal_equations = (features_centred.T @ features_centred
                            + self.alpha * np.eye(bin_features.shape[1]))
        self.coefficients = np.linalg.solve(normal_equations,
                                            features_centred.T @ velocity_centred)
        self.intercept = velocity_mean - feature_mean @ self.coefficients
        return self

    def predict(self, bin_features):
        return np.asarray(bin_features, np.float64) @ self.coefficients + self.intercept


class KNNRegressor:
    """k-nearest-neighbour regression on the same bin rows."""

    def __init__(self, n_neighbors):
        self.n_neighbors = n_neighbors

    def fit(self, bin_features, bin_velocity):
        self.model = KNeighborsRegressor(
            n_neighbors=min(self.n_neighbors, len(bin_features)), weights="distance"
        ).fit(bin_features, bin_velocity)
        return self

    def predict(self, bin_features):
        return self.model.predict(bin_features)

In [ ]:
import pandas as pd

LAG_COUNT = utils.LAG_COUNT
FEATURE_VARIANTS = ((utils.RAW_VARIANT, 'raw'), (utils.PARRM_VARIANT, 'parrm'))
DRAWS = utils.load_json(utils.DRAWS_PATH)


def clip(bin_rows):
    """Clamp z-scored features to +/-CLIP_Z so one artifact bin cannot dominate the weights.

    bin_rows is (bin_features (n_bins, n_features), bin_velocity (n_bins, 2)); only the
    features are clipped.
    """
    bin_features, bin_velocity = bin_rows
    return np.clip(bin_features, -task3_common.CLIP_Z, task3_common.CLIP_Z), bin_velocity


print('draws:', len(DRAWS['sessions']), 'sessions | ks', DRAWS['ks'],
      '| query definition', DRAWS.get('query_definition'))


draws: 14 sessions | ks [2, 5, 10, 20] | query definition remaining


## Protocol

Everything comes from `task3_common`, so task 3 and task 4.3 make identical choices and only
the fit data differs. That covers the ridge solver, the penalty grid
(`task3_common.LAMBDAS`), and the clipping (`task3_common.CLIP_Z`). Features are clipped to
$\pm 10$ so one artifact bin can't dominate the weights. The clip binds almost only on raw
features under stimulation, so if anything it handicaps the raw arm, and PARRM still doesn't
beat it.

$\lambda$ is picked once on the ON validation sessions and held for every draw. Each draw
normalizes on its own trials (see 4.1). Raw and PARRM share the same draws and query trials.
Each choice removes a confound, leaving $k$ and the input variant as the only things that
vary, which is the experiment.

In [ ]:
# OFF-training rows for choosing one penalty shared by every draw.
stimulation_off_train_bin_features, stimulation_off_train_bin_velocity = clip(
    utils.stack_session_bin_rows(utils.STIMULATION_OFF_TRAINING_SESSIONS, 'train',
                                 utils.RAW_VARIANT, LAG_COUNT))

# Validation rows are the query trials of the stimulated validation sessions.
validation_rows = [
    clip(utils.valid_bin_rows(
        utils.load_split_dataset(s, 'query', utils.RAW_VARIANT, LAG_COUNT)))
    for s in utils.sessions_in_split('val', utils.STIMULATION_ON)]

fitter = task3_common.RidgeFitter(stimulation_off_train_bin_features,
                                 stimulation_off_train_bin_velocity)

alpha_scores = {
    float(penalty): np.mean([
        utils.mean_r2(bin_velocity, fitter.fit(penalty).predict(bin_features))
        for bin_features, bin_velocity in validation_rows])
    for penalty in task3_common.LAMBDAS}

alpha = max(alpha_scores, key=alpha_scores.get)
print('ridge alpha:', alpha, '(from task 3 grid of', len(task3_common.LAMBDAS), ')')
pd.Series(alpha_scores).round(4)


ridge alpha: 100000.0 (from task 3 grid of 13 )


1.000000         -0.5760
3.162278         -0.5109
10.000000        -0.4159
31.622777        -0.3002
100.000000       -0.1876
316.227766       -0.0959
1000.000000      -0.0297
3162.277660       0.0090
10000.000000      0.0264
31622.776602      0.0315
100000.000000     0.0320
316227.766017     0.0311
1000000.000000    0.0270
dtype: float64

In [ ]:
def refit_draw(session, calibration_size, draw_index, draw_record,
               variant, feature_label, alpha, control_bin_rows):
    """Refit ridge on this draw's calibration bin rows and score its query bin rows.

    control_bin_rows is passed in because the same-day control recording does not depend on
    the draw; reloading it per draw re-reads the same control recording 100 times.
    """
    calibration_bin_features, calibration_bin_velocity = clip(
        utils.drawn_trial_windows(session, draw_record, variant, LAG_COUNT))
    query_bin_rows = clip(
        utils.query_trial_windows(session, draw_record, variant, LAG_COUNT))
    model = task3_common.ridge_path(
        calibration_bin_features, calibration_bin_velocity, [alpha])[0]  # one lambda
    return utils.score_window_decoder(model, session, 'ridge', feature_label,
                                      query_bin_rows, control_bin_rows,
                                      k=calibration_size, draw=draw_index)


score_rows = []
for session in DRAWS['sessions']:
    # Two control datasets per session (raw and PARRM), not one per draw.
    control_rows_by_label = {}
    for variant, feature_label in FEATURE_VARIANTS:
        control_dataset = utils.load_same_day_control_dataset(session, variant, LAG_COUNT)
        control_rows_by_label[feature_label] = clip(utils.valid_bin_rows(control_dataset))

    for calibration_size, draw_index, draw_record in utils.iter_draws(DRAWS, session):
        for variant, feature_label in FEATURE_VARIANTS:
            score_rows.append(refit_draw(
                session, calibration_size, draw_index, draw_record,
                variant, feature_label, alpha, control_rows_by_label[feature_label]))
    print('scored', session)

per_draw_scores = pd.DataFrame(score_rows)
per_draw_scores.to_csv(
    os.path.join(utils.TASK4_RESULTS, 'task4_3_parrm_ridge.csv'), index=False)
print(per_draw_scores.shape)
per_draw_scores.groupby(['feature_variant', 'k'])[
    ['r2_ave', 'r2_off_ave']].mean().round(3)


## Recovery per $k$

Each row is one (variant, $k$): the mean over 14 sessions, with a 95% CI that resamples
sessions and, inside each session, its calibration draws. The query trials are fixed for
every draw, so the interval covers session and calibration choice rather than query-set
luck. At $k=20$ the calibration set is the whole 20-trial pool and there is one draw, not
ten. `r2_off` is the OFF control from above.

Read the $k$ axis, not the gap between rows. A PARRM-vs-raw difference of 0.001 is
invisible at this scale, while the rise from $k=2$ to $k=20$ is 30-50x larger.

In [ ]:
summary_rows = []
for (variant, calibration_size), variant_slice in per_draw_scores.groupby(
        ['feature_variant', 'k']):
    per_session = [session_slice for _, session_slice in variant_slice.groupby('session')]
    per_session_r2 = [session_slice.r2_ave.to_numpy() for session_slice in per_session]
    per_session_off_r2 = [session_slice.r2_off_ave.to_numpy() for session_slice in per_session]

    low, high = utils.two_level_bootstrap_confidence_interval(per_session_r2)
    summary_rows.append(dict(
        feature_variant=variant, k=calibration_size, sessions=len(per_session_r2),
        draws_per_session=len(per_session_r2[0]),
        r2=float(np.mean([scores.mean() for scores in per_session_r2])),
        ci_low=low, ci_high=high,
        r2_off=float(np.mean([scores.mean() for scores in per_session_off_r2]))))

draw_summary = pd.DataFrame(summary_rows)
draw_summary.to_csv(os.path.join(utils.TASK4_RESULTS, 'task4_3_summary.csv'), index=False)
draw_summary.round(4)
